# Auslan → English: Arm B from `openasl_pose_only_slt.pth`

Two stages, in this order:

1. **B1 -- pose adaptation on MM-WLAuslan** (`Train`, `studio`, 38,580 isolated-sign clips).
   Only the pose side learns: the GCN pose encoders at `1e-4`, the temporal stack at `1e-5`.
   **The whole mT5 (`decoder` group, 582M) is frozen.** Targets are lower-cased glosses
   (`gloss_text_mode: strip_paren_lower`, e.g. `WHALE` -> `whale`).
2. **B2 -- Auslan-Daily fine-tuning** from the B1 weights, with the *exact* recipe of the
   finished OpenASL Arm A run: official Stage 3, `lr=3e-4`, effective batch 32, 20 epochs, BF16.

**One variable against OpenASL Arm A** (`arm_a__openasl_pose_only_slt__official_stage3__single_a100__bf16`,
Communication BLEU-4 18.03 / News 5.20, plain): whether B1 ran first. Starting checkpoint, data,
exclusions, optimiser, schedule, precision, seed and decoding are identical.

**Why the mT5 is frozen in B1.** The value of the OpenASL starting point is a decoder that already
writes English sentences. MM-WLAuslan targets are single glosses; three epochs of training the
decoder on one-word outputs could teach it to stop early. Freezing it forces B1 to adapt only what
MM-WLAuslan can plausibly help with -- how Auslan hand shapes and motion are encoded. (The old
CSL-based B1 kept the decoder at 1/10 lr; that run is not reused here.)

**B1 keeps the old B1 optimiser** (AdamW `1e-4`, weight decay `0.01`, batch 16, 3 epochs, 5% warmup,
cosine), only in BF16. It has no sentence-level validation set, so it produces a checkpoint, not a score.

**Why lower case.** MM-WLAuslan glosses are upper case, while Auslan-Daily references and the OpenASL
decoder are lower-case English. With the decoder frozen it cannot learn to write upper case, so upper-case
targets inflate the loss and push the pose encoder towards a format the decoder cannot produce. A first B1
with upper-case targets (`..._b1_pose_mt5frozen__bf16`) was stopped at ~7.5 loss and is not used.

**A cheap gate before the long part.** B2 runs epoch 0 first (about 10 minutes). Its mean loss is
compared with OpenASL Arm A's epoch 0 (5.48). If B1 damaged the representation, B2 starts clearly
higher, and the cell stops automatically instead of spending ~3 hours.

**GPU budget (A100, BF16):** B1 about 30-40 min, B2 about 3.4 h, validation decoding a few minutes.

Run the cells in order. After a Colab disconnect, re-run sections 1-6 and then the interrupted
training cell; every training cell uses `--resume` and skips finished work.

## 1. Runtime and dependencies

In [1]:
import subprocess
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip() or 'NO GPU')
DEVICE = 'cuda'

!pip -q install einops sacrebleu
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__, '| cuda', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit('CUDA is required for the real Uni-Sign run')

NVIDIA A100-SXM4-40GB, 40960 MiB
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.0/129.0 kB 11.7 MB/s eta 0:00:00
torch 2.11.0+cu128 | transformers 5.16.1 | cuda True


## 2. Mount Drive

In [2]:
import os, glob, json, copy, hashlib, shutil, signal, subprocess, sys, tarfile
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive'
WORK = f'{DRIVE}/auslan_work'
MANIFEST = f'{WORK}/manifest.jsonl'
EXCLUDE = f'{WORK}/excluded.txt'
for path in (MANIFEST, EXCLUDE):
    if not os.path.exists(path):
        raise SystemExit(f'{path} is missing; run colab_setup.ipynb first')
print('work dir:', WORK)

Mounted at /content/drive
work dir: /content/drive/MyDrive/auslan_work


## 3. Copy the verified project code

In [3]:
CODE = '/content/unisign'

def locate(parts):
    for root in (DRIVE, '/content'):
        for prefix in ('', '*/', '*/*/'):
            hits = glob.glob(os.path.join(root, prefix, *parts))
            if hits:
                return hits[0]
    return None

src = locate(['unisign', 'spec.py'])
if src:
    src = os.path.dirname(src)
else:
    tarball = locate(['unisign_code.tar.gz'])
    if tarball is None:
        raise SystemExit('Upload unisign/ or unisign_code.tar.gz to Drive first')
    with tarfile.open(tarball) as tf:
        tf.extractall('/content/_code')
    src = '/content/_code/unisign'
if os.path.abspath(src) != CODE:
    shutil.rmtree(CODE, ignore_errors=True)
    shutil.copytree(src, CODE)
sys.path.insert(0, CODE)
import spec
print('code from', src)
print('spec fingerprint', spec.SCHEMA_FINGERPRINT)
assert spec.SCHEMA_FINGERPRINT == 'bc3bb2df0f22948d', 'spec.py does not match extracted poses'
_train_py = open(f'{CODE}/train.py').read()
for needed, why in [('gradient_accumulation_steps', 'gradient accumulation'),
                    ('--stop-after-epochs', 'the epoch-0 pause'),
                    ('autocast_context', 'precision: bf16')]:
    assert needed in _train_py, f'This train.py has no support for {why}; upload the current unisign_code.tar.gz first'
print('train.py supports gradient accumulation, --stop-after-epochs and bf16')

code from /content/drive/MyDrive/unisign
spec fingerprint bc3bb2df0f22948d
train.py supports gradient accumulation, --stop-after-epochs and bf16


## 4. Download the pinned Uni-Sign model and mT5

Same commit, revisions and sha256 as the OpenASL Arm A run.

In [4]:
from huggingface_hub import hf_hub_download, snapshot_download

INIT_CKPT = 'openasl_pose_only_slt.pth'
REPO_DIR = '/content/Uni-Sign'
REPO_COMMIT = 'eed438bcb49e30405cd6ccdfcccca330c134e830'
MT5_DIR = f'{REPO_DIR}/pretrained_weight/mt5-base'
MT5_REVISION = '2eb15465c5dd7f72a8f7984306ad05ebc3dd1e1f'
UNISIGN_REVISION = 'eab251b7fe7e8521afc0e67be98add670ea40a0d'
SHA256 = {
    'openasl_pose_only_slt.pth': 'f836ea66bc837bbe6ed717a4b9bece87875f03ef96d4bf1092ca3dd767982798',
    'mt5-base/pytorch_model.bin': '180573b534144580f04af026da62bf71bc976ee1b7eb311b8945e2fefde8d614',
}

if not os.path.isdir(f'{REPO_DIR}/.git'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/ZechengLi19/Uni-Sign.git', REPO_DIR], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'checkout', '-q', REPO_COMMIT], check=True)
snapshot_download('google/mt5-base', revision=MT5_REVISION, local_dir=MT5_DIR, allow_patterns=['*.json', '*.model', 'pytorch_model.bin'])
CKPT = hf_hub_download('ZechengLi19/Uni-Sign', INIT_CKPT, revision=UNISIGN_REVISION, local_dir='/content/checkpoints')

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 24), b''):
            h.update(block)
    return h.hexdigest()

for name, path in [(INIT_CKPT, CKPT), ('mt5-base/pytorch_model.bin', f'{MT5_DIR}/pytorch_model.bin')]:
    if sha256(path) != SHA256[name]:
        raise SystemExit(f'{name}: sha256 mismatch')
    print('OK', name)
print('Uni-Sign code at', REPO_COMMIT[:7])

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

openasl_pose_only_slt.pth: reconstructing file:   0%|          |  0.00B / 1.19GB            

openasl_pose_only_slt.pth: downloading bytes:           |  0.00B            

OK openasl_pose_only_slt.pth
OK mt5-base/pytorch_model.bin
Uni-Sign code at eed438b


## 5. Unpack poses and gate the data

B1 needs MM-WLAuslan poses and B2 needs Auslan-Daily poses, so this gate checks both.

`excluded.txt` must be exactly the 131 Auslan-Daily uids used by OpenASL Arm A. If it was overwritten
(e.g. by the MM-WLAuslan verify pass), it is repaired from the protected copy
`excluded_auslan_daily.txt` written by `colab_train_bc.ipynb`; if neither is valid the cell stops.
A list containing `mmwl-` uids would silently drop B1 data, so that also stops the run.

In [5]:
import yaml
POSE_LOCAL = '/content/pose'
os.makedirs(POSE_LOCAL, exist_ok=True)
rows = [json.loads(line) for line in open(MANIFEST)]
ad_rows = [r for r in rows if r['dataset'] == 'auslandaily']
b1_rows = [r for r in rows if r['dataset'] == 'mmwlauslan' and r['split'] == 'train' and r.get('subset') == 'studio']
print(f'{len(rows)} manifest rows | {len(ad_rows)} auslandaily | {len(b1_rows)} mmwlauslan train/studio')
if len(ad_rows) != 25109 or len(b1_rows) != 38580:
    raise SystemExit('manifest.jsonl is not the joint manifest (expected 25109 auslandaily and 38580 mmwlauslan train/studio rows)')

needed = ad_rows + b1_rows
if len(glob.glob(f'{POSE_LOCAL}/*.npz')) < len(needed):
    chunks = sorted(glob.glob(f'{WORK}/pose/chunk_*.tar'))
    print(f'unpacking {len(chunks)} chunk(s) from Drive ...')
    for chunk in chunks:
        with tarfile.open(chunk) as tf:
            tf.extractall(POSE_LOCAL, filter='data')
present = {os.path.basename(path)[:-4] for path in glob.glob(f'{POSE_LOCAL}/*.npz')}
missing = [r['uid'] for r in needed if r['uid'] not in present]
print(f'{len(present)} poses on disk | {len(missing)} missing among B1+B2 clips')
if missing:
    raise SystemExit(f'{len(missing)} clips have no pose, e.g. {missing[:3]}')

EXPECTED_AD_EXCLUSIONS = 131
AD_EXCLUDE = f'{WORK}/excluded_auslan_daily.txt'

def read_ids(path):
    return {line.strip() for line in open(path) if line.strip() and not line.startswith('#')}

def valid_gate(ids):
    return len(ids) == EXPECTED_AD_EXCLUSIONS and all(u.startswith('ad-') for u in ids)

excluded = read_ids(EXCLUDE)
if not valid_gate(excluded):
    protected = read_ids(AD_EXCLUDE) if os.path.exists(AD_EXCLUDE) else set()
    if not valid_gate(protected):
        raise SystemExit(
            f'excluded.txt holds {len(excluded)} uids ({sum(u.startswith("ad-") for u in excluded)} ad-), '
            f'expected exactly {EXPECTED_AD_EXCLUSIONS} ad- uids, and no valid protected copy exists. '
            'Regenerate it with verify_pose.py --exclude-out over the Auslan-Daily poses.')
    shutil.copyfile(AD_EXCLUDE, EXCLUDE)
    excluded = protected
    print('excluded.txt repaired from', AD_EXCLUDE)
ad_train_val = {r['uid'] for r in ad_rows if r['split'] in ('train', 'val')}
print(f'exclusions: {len(excluded)} ad- uids | {len(excluded & ad_train_val)} in train/val | 0 mmwl-')

76549 manifest rows | 25109 auslandaily | 38580 mmwlauslan train/studio
unpacking 477 chunk(s) from Drive ...
76549 poses on disk | 0 missing among B1+B2 clips
exclusions: 131 ad- uids | 119 in train/val | 0 mmwl-


## 6. Write the B1 and B2 configs

- **B1** keeps the old stage-1 optimiser; `trainable_groups` leaves out `decoder`, so every mT5
  parameter has `requires_grad=False` and is not in the optimiser. Gradients still flow *through*
  the frozen mT5 into the temporal stack and pose encoders.
- **B2** copies the OpenASL Arm A optimiser block key by key. `--init-from` loads the B1 weights on
  top of the backend checkpoint, so the backend still names `openasl_pose_only_slt.pth` (it only
  provides the architecture and the tokenizer path).

Watch the B1 log for `decoder ... lr=FROZEN`, and the B2 log for `587.75M / 587.75M` and
`initialised from .../checkpoint.pt`.

In [6]:
STEM = INIT_CKPT.rsplit('.', 1)[0]
RUN_B1 = f'arm_b__{STEM}__b1_pose_mt5frozen_lower__bf16'
RUN_B2 = f'arm_b__{STEM}__b2_official_stage3__single_a100__bf16'
RUN_ARM_A = f'arm_a__{STEM}__official_stage3__single_a100__bf16'   # the reference B must beat

BACKEND = {'name': 'unisign', 'checkpoint': CKPT, 'repo': REPO_DIR, 'mt5_path': MT5_DIR,
           'num_beams': 4, 'max_new_tokens': 100, 'label_smoothing': 0.2}

def write_cfg(template, run, optim, log_every, **data_overrides):
    cfg = yaml.safe_load(open(f'{CODE}/configs/{template}.yaml'))
    cfg['seed'] = 0
    cfg['num_workers'] = 8
    cfg['log_every'] = log_every
    cfg['data'].update(manifest=MANIFEST, npz_dir=POSE_LOCAL, exclude=EXCLUDE, max_length=256, **data_overrides)
    cfg['backend'] = dict(BACKEND)
    cfg['optim'] = optim
    cfg['decode'] = {}
    cfg['precision'] = 'bf16'
    cfg['output_dir'] = f'{WORK}/runs/{run}'
    path = f'{WORK}/train_configs/{run}.yaml'
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w') as fh:
        yaml.safe_dump(cfg, fh, sort_keys=False)
    return path, cfg

B1_OPTIM = dict(lr=1e-4, weight_decay=0.01, batch_size=16, epochs=3, warmup_frac=0.05,
                grad_clip=1.0, gradient_accumulation_steps=1,
                trainable_groups=['pose_encoder', 'temporal'],        # mT5 frozen
                lr_scale={'pose_encoder': 1.0, 'temporal': 0.1})
# Identical to OpenASL Arm A (colab_train_openasl_init.ipynb, section 5).
B2_OPTIM = dict(lr=3e-4, weight_decay=1e-4, batch_size=8, epochs=20, warmup_frac=0.0,
                grad_clip=1.0, trainable_groups=None, gradient_accumulation_steps=4,
                betas=[0.9, 0.999], eps=1e-9)

# MM-WLAuslan glosses are upper case ("WHALE"); the frozen English decoder writes lower case.
# Lower-casing the B1 targets keeps the gradient about the sign, not about the casing.
CFG_B1, CONFIG_B1 = write_cfg('arm_b_stage1', RUN_B1, B1_OPTIM, log_every=100,
                              gloss_text_mode='strip_paren_lower')
assert CONFIG_B1['data']['gloss_text_mode'] == 'strip_paren_lower'
CFG_B2, CONFIG_B2 = write_cfg('arm_b_stage2', RUN_B2, B2_OPTIM, log_every=50)
OUT_B1, OUT_B2 = CONFIG_B1['output_dir'], CONFIG_B2['output_dir']
B1_WEIGHTS = f'{OUT_B1}/checkpoint.pt'
METRICS_B2 = f'{OUT_B2}/metrics.json'
LOG_B2 = f'{OUT_B2}/train_log.jsonl'

# If the Arm A config is on Drive, prove the B2 optimiser really matches it.
arm_a_cfg_path = f'{WORK}/train_configs/{RUN_ARM_A}.yaml'
if os.path.exists(arm_a_cfg_path):
    arm_a_cfg = yaml.safe_load(open(arm_a_cfg_path))
    for key in ('optim', 'backend', 'precision', 'seed'):
        a = arm_a_cfg.get(key)
        b = CONFIG_B2.get(key)
        if key == 'backend':
            a = {k: v for k, v in a.items() if k != 'checkpoint'}
            b = {k: v for k, v in b.items() if k != 'checkpoint'}
        if a != b:
            raise SystemExit(f'B2 {key} differs from OpenASL Arm A: {a} vs {b}')
    print('B2 optimiser, backend, precision and seed match', arm_a_cfg_path)
else:
    print('note: Arm A config not found on Drive; B2 uses the recipe copied from its notebook')

print('B1:', RUN_B1)
print(yaml.safe_dump({'train': CONFIG_B1['train'], 'gloss_text_mode': CONFIG_B1['data']['gloss_text_mode'], 'optim': CONFIG_B1['optim']}, sort_keys=False))
print('B2:', RUN_B2)
print(yaml.safe_dump({'train': CONFIG_B2['train'], 'val': CONFIG_B2['val'], 'gloss_text_mode': CONFIG_B2['data']['gloss_text_mode'], 'optim': CONFIG_B2['optim']}, sort_keys=False))

B2 optimiser, backend, precision and seed match /content/drive/MyDrive/auslan_work/train_configs/arm_a__openasl_pose_only_slt__official_stage3__single_a100__bf16.yaml
B1: arm_b__openasl_pose_only_slt__b1_pose_mt5frozen_lower__bf16
train:
  datasets:
  - mmwlauslan
  subsets:
  - studio
  splits:
  - train
gloss_text_mode: strip_paren_lower
optim:
  lr: 0.0001
  weight_decay: 0.01
  batch_size: 16
  epochs: 3
  warmup_frac: 0.05
  grad_clip: 1.0
  gradient_accumulation_steps: 1
  trainable_groups:
  - pose_encoder
  - temporal
  lr_scale:
    pose_encoder: 1.0
    temporal: 0.1

B2: arm_b__openasl_pose_only_slt__b2_official_stage3__single_a100__bf16
train:
  datasets:
  - auslandaily
  splits:
  - train
val:
  datasets:
  - auslandaily
  splits:
  - val
gloss_text_mode: strip_paren
optim:
  lr: 0.0003
  weight_decay: 0.0001
  batch_size: 8
  epochs: 20
  warmup_frac: 0.0
  grad_clip: 1.0
  trainable_groups: null
  gradient_accumulation_steps: 4
  betas:
  - 0.9
  - 0.999
  eps: 1.0e-09


## 7. Training helper

Streams the log, and on Stop sends Ctrl-C so `train.py` writes a resumable checkpoint (exit code 130).

In [7]:
def run_train(cfg_path, extra, log_path=None):
    cmd = [sys.executable, '-u', 'train.py', '--config', cfg_path, '--device', DEVICE] + extra
    log = open(log_path, 'a') if log_path else None
    proc = subprocess.Popen(cmd, cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in proc.stdout:
            print(line, end='', flush=True)
            if log:
                log.write(line); log.flush()
    except KeyboardInterrupt:
        print('Stop pressed: train.py will save a checkpoint ...', flush=True)
        proc.send_signal(signal.SIGINT)
        for line in proc.stdout:
            print(line, end='', flush=True)
            if log:
                log.write(line); log.flush()
    proc.wait()
    if log:
        log.close()
    return proc.returncode

## 8. B1: MM-WLAuslan pose adaptation (mT5 frozen)

About 30-40 minutes. Deliverable: `checkpoint.pt`. The loss here is gloss prediction through a frozen
English decoder; it should fall, but its absolute value is not a result.

In [8]:
import statistics
if os.path.exists(B1_WEIGHTS):
    print('B1 already finished:', B1_WEIGHTS)
else:
    os.makedirs(OUT_B1, exist_ok=True)
    rc = run_train(CFG_B1, ['--resume'], log_path=f'{OUT_B1}/train.log')
    if rc == 130:
        raise SystemExit('B1 stopped safely; re-run this cell to continue')
    if rc != 0:
        raise RuntimeError(f'B1 exited with code {rc}; checkpoint is retained')
if not os.path.exists(B1_WEIGHTS):
    raise SystemExit(f'{B1_WEIGHTS} was not written')
b1_log = [json.loads(l) for l in open(f'{OUT_B1}/train_log.jsonl')]
for ep in sorted({r['epoch'] for r in b1_log}):
    print(f"B1 epoch {ep}: mean loss {statistics.fmean(r['loss'] for r in b1_log if r['epoch'] == ep):.3f}")

arm=arm_b_stage1 device=cuda precision=bf16 out=/content/drive/MyDrive/auslan_work/runs/arm_b__openasl_pose_only_slt__b1_pose_mt5frozen_lower__bf16
train=38580
left out 0 clips listed in /content/drive/MyDrive/auslan_work/excluded.txt

Loading weights: 100%|██████████| 284/284 [00:00<00:00, 14603.55it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[load] missing=0 unexpected=0
trainable groups=['pose_encoder', 'temporal'] 5.35M / 587.75M (0.9%)
    decoder         582.40M  lr=FROZEN
    pose_encoder      0.41M  lr=1.00e-04
    temporal          4.94M  lr=1.00e-05
batch=16 gradient_accumulation=1 effective_batch=16 updates/epoch=2411
--resume: no checkpoint in /content/drive/MyDrive/auslan_work/runs/arm_b__openasl_pose_only_slt__b1_pose_mt5froz

## 9. B2 epoch 0

Starts from the B1 weights and stops at the first epoch boundary (about 10 minutes). The pause does not
change training: the cosine schedule still spans 20 epochs and `--stop-after-epochs` is not in the
resume fingerprint.

In [9]:
if not os.path.exists(B1_WEIGHTS):
    raise SystemExit('finish B1 first')
if os.path.exists(METRICS_B2):
    print('B2 already finished:', OUT_B2)
else:
    os.makedirs(OUT_B2, exist_ok=True)
    rc = run_train(CFG_B2, ['--resume', '--init-from', B1_WEIGHTS, '--stop-after-epochs', '1'],
                   log_path=f'{OUT_B2}/train.log')
    if rc not in (0, 130):
        raise RuntimeError(f'B2 exited with code {rc}; checkpoint is retained')
    print('B2 epoch 0 done' if rc == 130 else 'B2 already past epoch 0')

arm=arm_b_stage2 device=cuda precision=bf16 out=/content/drive/MyDrive/auslan_work/runs/arm_b__openasl_pose_only_slt__b2_official_stage3__single_a100__bf16
train=21998 val=1492
left out 119 clips listed in /content/drive/MyDrive/auslan_work/excluded.txt

Loading weights: 100%|██████████| 284/284 [00:00<00:00, 15630.67it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[load] missing=0 unexpected=0
initialised from /content/drive/MyDrive/auslan_work/runs/arm_b__openasl_pose_only_slt__b1_pose_mt5frozen_lower__bf16/checkpoint.pt (arm=arm_b_stage1 step=7233)
trainable groups=['decoder', 'pose_encoder', 'temporal'] 587.75M / 587.75M (100.0%)
    decoder         582.40M  lr=3.00e-04
    pose_encoder      0.41M  lr=3.00e-04
    temporal          4.94M 

## 9b. Epoch-0 gate (automatic)

Reference: OpenASL Arm A averaged **5.48** over epoch 0 with the same recipe. B2 differs only by the
B1 weights, so:

- **mean ≤ 5.78** (not more than +0.3): B1 did not break anything -- continue automatically.
- **mean > 5.78**: B1 damaged the representation, and the remaining ~3 hours would most likely end below
  Arm A. The next cell refuses to run. Set `FORCE_CONTINUE = True` there only if you decide otherwise.

A lower epoch-0 loss than Arm A is encouraging but is not a result; judge on the validation metrics.

In [10]:
ARM_A_EPOCH0 = 5.48      # OpenASL Arm A, epoch-0 mean over 13 logged points
GATE_BAND = 0.3
b2_log = [json.loads(l) for l in open(LOG_B2)] if os.path.exists(LOG_B2) else []
ep0 = [r['loss'] for r in b2_log if r['epoch'] == 0]
if not ep0:
    raise SystemExit('no B2 epoch-0 entries yet; run the previous cell')
mean0 = statistics.fmean(ep0)
delta0 = mean0 - ARM_A_EPOCH0
print(f'B2 epoch 0: {len(ep0)} points | first {ep0[0]:.2f} | last {ep0[-1]:.2f} | mean {mean0:.2f}')
print(f'OpenASL Arm A epoch 0 mean: {ARM_A_EPOCH0} | difference {delta0:+.2f}')
B2_GATE_OK = delta0 <= GATE_BAND
print('-> gate passed: continue to epochs 1-19' if B2_GATE_OK else
      '-> gate failed: B1 made the start worse; epochs 1-19 will not run automatically')

B2 epoch 0: 13 points | first 5.77 | last 5.42 | mean 5.50
OpenASL Arm A epoch 0 mean: 5.48 | difference +0.02
-> gate passed: continue to epochs 1-19


## 10. B2 epochs 1-19

About 3.3 hours, then validation decoding with plain beam search (`num_beams=4`). Re-run after a
disconnect; it resumes and exits immediately once finished.

In [11]:
FORCE_CONTINUE = False   # only to override a failed epoch-0 gate on purpose

if os.path.exists(METRICS_B2):
    print('B2 already finished:', OUT_B2)
elif not (B2_GATE_OK or FORCE_CONTINUE):
    raise SystemExit('epoch-0 gate failed; see 9b. Checkpoint kept at ' + f'{OUT_B2}/checkpoints')
else:
    rc = run_train(CFG_B2, ['--resume', '--init-from', B1_WEIGHTS], log_path=f'{OUT_B2}/train.log')
    if rc == 130:
        raise SystemExit('Stopped safely; reconnect, re-run sections 1-7 and this cell')
    if rc != 0:
        raise RuntimeError(f'B2 exited with code {rc}; checkpoint is retained')
    print('metrics:', METRICS_B2)

arm=arm_b_stage2 device=cuda precision=bf16 out=/content/drive/MyDrive/auslan_work/runs/arm_b__openasl_pose_only_slt__b2_official_stage3__single_a100__bf16
train=21998 val=1492
left out 119 clips listed in /content/drive/MyDrive/auslan_work/excluded.txt

Loading weights: 100%|██████████| 284/284 [00:00<00:00, 16197.97it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[load] missing=0 unexpected=0
trainable groups=['decoder', 'pose_encoder', 'temporal'] 587.75M / 587.75M (100.0%)
    decoder         582.40M  lr=3.00e-04
    pose_encoder      0.41M  lr=3.00e-04
    temporal          4.94M  lr=3.00e-04
batch=8 gradient_accumulation=4 effective_batch=32 updates/epoch=687
resumed from ckpt_step000000687.pt: step 687/13740, epoch 1 batch 0
  {"step":

## 11. Compare with Arm A

The row that matters is **openasl B vs openasl A**: one variable. The CSL rows are context.
Noise floor from two old seeds: BLEU-4 0.63 (Communication) and 0.22 (News). It was measured under a
different recipe and B/A are single seeds, so treat a gap near that size as inconclusive.

In [12]:
subprocess.run([sys.executable, 'plot_loss.py', LOG_B2], cwd=CODE, check=False)
RUNS = [
    ('csl A fp32', 'arm_a__csl_stage1_weight__official_stage3__single_a100'),
    ('csl A bf16', 'arm_a__csl_stage1_weight__official_stage3__single_a100__bf16'),
    ('csl B2 (old)', 'arm_bc__csl_stage1_weight__b2_daily'),
    ('openasl A', RUN_ARM_A),
    ('openasl B', RUN_B2),
]
FIELDS = ['BLEU-1', 'BLEU-4', 'ROUGE-L', 'unique_hyps', 'looping', 'hyp_len']
print(f"{'run':<14}{'subset':<15}{'n':>5}" + ''.join(f'{f:>12}' for f in FIELDS))
table = {}
for label, run in RUNS:
    path = f'{WORK}/runs/{run}/metrics.json'
    if not os.path.exists(path):
        print(f'{label:<14}(not finished)')
        continue
    table[label] = json.load(open(path))
    for key, m in table[label].items():
        print(f"{label:<14}{key.split('/')[-1]:<15}{m.get('n', 0):>5}"
              + ''.join(f"{m.get(f, float('nan')):>12}" for f in FIELDS))

NOISE = {'communication': 0.63, 'news': 0.22}
if 'openasl A' in table and 'openasl B' in table:
    print('\nopenasl B - openasl A (plain):')
    for key in table['openasl A']:
        sub = key.split('/')[-1]
        a, b = table['openasl A'][key], table['openasl B'][key]
        d4 = b['BLEU-4'] - a['BLEU-4']
        verdict = 'B better' if d4 > NOISE[sub] else 'B worse' if d4 < -NOISE[sub] else 'inside the noise floor'
        print(f"  {sub:<14} BLEU-4 {d4:+.2f}  ROUGE-L {b['ROUGE-L'] - a['ROUGE-L']:+.2f}  -> {verdict}")
print('\nSubsets are reported separately; no average.')

run           subset             n      BLEU-1      BLEU-4     ROUGE-L unique_hyps     looping     hyp_len
csl A fp32    communication    792       36.12       11.51       23.78        51.6         1.8         5.1
csl A fp32    news             700       17.39        2.17        12.9        78.0        34.6        18.6
csl A bf16    communication    792       33.62        9.77       20.55        40.9         2.4         5.1
csl A bf16    news             700       16.85        1.82       12.33        65.1        37.9        18.4
csl B2 (old)  communication    792       26.33         3.4       12.49         9.1         2.1         4.9
csl B2 (old)  news             700        6.86        0.54        6.23        38.3        66.6        29.0
openasl A     communication    792       43.99       18.03       31.67        71.7         0.6         4.9
openasl A     news             700       26.01         5.2       18.86        97.3        12.9        15.9
openasl B     communication    792   

## How to read the outcome

- **B better than A on a subset by more than the noise floor**: MM-WLAuslan pose adaptation helps on top of
  the English starting point. B becomes the reference, and Arm C (auxiliary loss from the same starting
  point) is worth one run.
- **Inside the noise floor**: B1 costs ~40 minutes and buys nothing measurable; keep Arm A as the reference.
  Arm C is then unlikely to be worth its GPU time either.
- **B worse**: the isolated-sign stage pulls the encoder away from continuous signing. Do not run Arm C with
  the same data before rethinking it.
- News OOV (10.2% word types) is a property of the data; neither A nor B is expected to change it.

All figures are single-seed validation scores under the exclusion list. Final test-set numbers need the
decoding setting fixed in advance (plain) and must not apply `excluded.txt`.